# HAM10000 / ISIC-Style EfficientNet-B0 Training

Research-only. Not for diagnosis, treatment decisions, or clinical deployment. Doctor review required.

This notebook is separate from the DermaCon-IN OPD model. Use it for dermoscopy-style lesion labels such as `mel`, `nv`, `bcc`, `bkl`, `akiec`, `df`, `vasc`.


## 1. Runtime

In Colab, choose `Runtime -> Change runtime type -> GPU`.

This notebook is runtime-only by default. No Google Drive is required. If the runtime resets, downloaded data and outputs are lost unless you download the artifacts at the end.


In [ ]:
!nvidia-smi


## 2. Repo And Runtime Paths


In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/Abhigyan-Shekhar/skin-lesion-detect-model.git'
REPO_DIR = Path('/content/skin-lesion-detect-model')
DATA_ROOT = Path('/content/ham10000')
ARTIFACT_ROOT = Path('/content/ham10000_artifacts')
DATASET_SLUG = 'kmader/skin-cancer-mnist-ham10000'

DATA_ROOT.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

print('REPO_DIR =', REPO_DIR)
print('DATA_ROOT =', DATA_ROOT)
print('ARTIFACT_ROOT =', ARTIFACT_ROOT)
print('Kaggle dataset =', DATASET_SLUG)


## 3. Clone Repo And Install Dependencies


In [ ]:
import subprocess
from pathlib import Path

def run_checked(cmd, *, cwd=None):
    print('+', ' '.join(cmd))
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr)
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")
    return result

if not REPO_DIR.exists():
    run_checked(['git', 'clone', REPO_URL, str(REPO_DIR)])
else:
    run_checked(['git', 'pull'], cwd=REPO_DIR)

%cd {REPO_DIR}
!pip install -q -r requirements.txt kaggle


## 4. Kaggle Credentials

Upload your `kaggle.json` API token file from Kaggle: `Account -> Create New Token`.

If `/root/.config/kaggle/kaggle.json` already exists in this runtime, you can skip the upload cell.


In [ ]:
from pathlib import Path

kaggle_config = Path('/root/.config/kaggle')
kaggle_token = kaggle_config / 'kaggle.json'
kaggle_config.mkdir(parents=True, exist_ok=True)

if kaggle_token.exists():
    print('Found existing Kaggle token:', kaggle_token)
else:
    from google.colab import files
    uploaded = files.upload()
    if 'kaggle.json' not in uploaded:
        raise FileNotFoundError('Upload kaggle.json to continue.')
    kaggle_token.write_bytes(uploaded['kaggle.json'])
    print('Saved Kaggle token to', kaggle_token)

!chmod 600 /root/.config/kaggle/kaggle.json
!ls -l /root/.config/kaggle


## 5. Download HAM10000 From Kaggle

Default source: [Skin Cancer MNIST: HAM10000](https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000).

This cell skips the download if the expected metadata file and image folders already exist in `/content/ham10000`.


In [ ]:
from pathlib import Path

metadata_path = DATA_ROOT / 'HAM10000_metadata.csv'
part1 = DATA_ROOT / 'HAM10000_images_part_1'
part2 = DATA_ROOT / 'HAM10000_images_part_2'

if metadata_path.exists() and part1.exists() and part2.exists():
    print('Using existing HAM10000 files in', DATA_ROOT)
else:
    !kaggle datasets download -d $DATASET_SLUG -p $DATA_ROOT --unzip

print('Metadata exists:', metadata_path.exists(), metadata_path)
print('Part 1 exists:', part1.exists(), part1)
print('Part 2 exists:', part2.exists(), part2)
!find /content/ham10000 -maxdepth 2 -type f | head -40


## 6. Prepare HAM10000 Splits

This uses the repo split-prep script and writes train/val/test CSVs under `data/ham10000/splits`.

Current implementation uses image-level stratified splitting. HAM10000 includes `lesion_id`; for stricter leakage control, grouped lesion-level splitting should be used before reporting final research metrics.


In [ ]:
!python src/prepare_ham10000.py \
  --metadata /content/ham10000/HAM10000_metadata.csv \
  --image_dirs /content/ham10000/HAM10000_images_part_1 /content/ham10000/HAM10000_images_part_2 \
  --output_dir data/ham10000/splits

!cat data/ham10000/splits/split_summary.json


## 7. Train HAM10000 EfficientNet-B0


In [ ]:
!python src/train.py --config configs/efficientnet_b0_ham10000.yaml


## 8. Evaluate Best Checkpoint


In [ ]:
!python src/evaluate.py \
  --checkpoint outputs_ham10000/checkpoints/best.pt \
  --split data/ham10000/splits/test.csv \
  --output_dir outputs_ham10000

!cat outputs_ham10000/metrics/metrics.json
!ls -R outputs_ham10000 | sed -n '1,160p'


## 9. Inference Test

You can upload a dermoscopy image and run inference with the trained HAM10000 model.


In [ ]:
from google.colab import files
uploaded = files.upload()

image_path = next(iter(uploaded.keys()))
!python src/inference.py --checkpoint outputs_ham10000/checkpoints/best.pt --image "$image_path" --top_k 5


## 10. Export Artifacts

This bundles the trained checkpoints, metrics, plots, and predictions into a zip you can download from Colab.


In [ ]:
from pathlib import Path
import shutil

export_dir = ARTIFACT_ROOT / 'outputs_ham10000'
if export_dir.exists():
    shutil.rmtree(export_dir)
shutil.copytree(REPO_DIR / 'outputs_ham10000', export_dir)

archive_base = ARTIFACT_ROOT / 'outputs_ham10000'
archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=ARTIFACT_ROOT, base_dir='outputs_ham10000')
print('Created:', archive_path)
!find /content/ham10000_artifacts -maxdepth 2 -type f | sort

from google.colab import files
files.download(archive_path)
